In [ ]:
## Step 1 — Load CSVs and replace missing values (999, 777) with NaN

import pandas as pd
import numpy as np
import os

os.chdir(r'C:\Users\cigde\Dropbox\Habituation_paper\October_Emily\input_arrays\pls_regression')
         
# Load files and make unique ID number inputs as string (convert from numbers)
demog = pd.read_csv('demog_clin_data.csv', dtype={'UniqueID': str})
spectral = pd.read_csv('spectral_RS_all_ages.csv', dtype={'UniqueID': str})
component = pd.read_csv('component_RS_all_ages.csv', dtype={'UniqueID': str})
cluster = pd.read_csv('cluster_RS_all_ages.csv', dtype={'UniqueID': str})


# Filter demog - remove ID-ASD and ID-control
demog = demog[~demog['group_original'].isin(['ID-ASD', 'ID-control'])].copy()
print(f"Demog after filtering: {demog.shape}")
print(demog['group_original'].value_counts())


# Replace 999 and 777 with NaN
demog = demog.replace([999, 777], np.nan)
spectral = spectral.replace([999, 777], np.nan)
component = component.replace([999, 777], np.nan)
cluster = cluster.replace([999, 777], np.nan)

for name, d in [('spectral', spectral), ('component', component), ('cluster', cluster)]:
    print(name, d.shape, d['UniqueID'].nunique())
    print(d.columns.tolist())

    
# NOTE: in the spectral analyses three files were excluded due to abnomral PSD so in the end we have:
#52 X TD child
#65 X ASD child (was 66, one removed) - number 4 ID '155192407573'
#69 X TD adolescent
#73 X ASD adolescent
#50 X TD adult (was 52, two removed) - number 10 ID '391792121294' AND number 15 ID '505506013383' 
# 63 X ASD adult

# These three recordings need to be removed from component and cluster arrays too

In [ ]:
# Step 2a:Remove from component and cluster three outliers who is not in spectral array

keep = set(spectral['UniqueID'])
print(sorted(set(component['UniqueID']) - keep))
component = component[component['UniqueID'].isin(keep)].reset_index(drop=True)
cluster = cluster[cluster['UniqueID'].isin(keep)].reset_index(drop=True)
print(spectral.shape, component.shape, cluster.shape)


In [ ]:
# Step 2b: Inspect subject ID column names and formats across all files

print("=== DEMOG ===")
print(demog.columns.tolist())
print(demog.iloc[0, 0], type(demog.iloc[0, 0]))

print("=== SPECTRAL ===")
print(spectral.columns.tolist())
print(spectral.iloc[0, 0], type(spectral.iloc[0, 0]))

print("=== COMPONENT ===")
print(component.columns.tolist())
print(component.iloc[0, 0], type(component.iloc[0, 0]))

print("=== CLUSTER ===")
print(cluster.columns.tolist())
print(cluster.iloc[0, 0], type(cluster.iloc[0, 0]))
print(cluster.iloc[0, 1], type(cluster.iloc[0, 1]))

In [ ]:
# Step 3: Sort all EEG files by UniqueID ascending and drop SubjectID

# Note: spectral and component were generated by concatenating two cohorts 
# (MADE_TD/ASD_habit_1 and MADE_TD/ASD_habit_2 concatanated to MADE_TD/ASD_habit_all)
# after this concantanetaion the order is not ascending)so subjects 
# are not in ascending Unique_ID order - sorting here fixes this
# cluster was already in ascending Unique_ID order as it was generated from std files

spectral = spectral.drop(columns=['SubjectID']).sort_values('UniqueID').reset_index(drop=True)
component = component.drop(columns=['SubjectID']).sort_values('UniqueID').reset_index(drop=True)
cluster = cluster.drop(columns=['SubjectID']).sort_values('UniqueID').reset_index(drop=True)

# Verify first 5 UniqueIDs match across files
print("=== FIRST 5 UniqueIDs ===")
print("spectral:", spectral['UniqueID'].head().tolist())
print("component:", component['UniqueID'].head().tolist())
print("cluster:", cluster['UniqueID'].head().tolist())

# Verify all three have same subjects
print(f"\nSpectral subjects: {spectral['UniqueID'].nunique()}")
print(f"Component subjects: {component['UniqueID'].nunique()}")
print(f"Cluster subjects: {cluster['UniqueID'].nunique()}")
print(f"\nAll three match: {set(spectral['UniqueID']) == set(component['UniqueID']) == set(cluster['UniqueID'])}")

# Save as ordered clean versions
save_path = r'C:\Users\cigde\Dropbox\Habituation_paper\October_Emily\input_arrays\pls_regression'
spectral.to_csv(os.path.join(save_path, 'spectral_RS_ord.csv'), index=False)
component.to_csv(os.path.join(save_path, 'component_RS_ord.csv'), index=False)
cluster.to_csv(os.path.join(save_path, 'cluster_RS_ord.csv'), index=False)
print("\nSaved ordered files")


In [ ]:
## Step 4: filter demog to keep only subjects whose UniqueID appears in the EEG files

demog = demog[demog['UniqueID'].isin(spectral['UniqueID'])].reset_index(drop=True)
print(f"Demog after filtering to EEG subjects: {demog.shape}")
print(demog['group_original'].value_counts())

# Check all four files have matching UniqueIDs before merging
eeg_ids = set(spectral['UniqueID'])
print(f"Spectral:  {len(spectral['UniqueID'].unique())} subjects")
print(f"Component: {len(component['UniqueID'].unique())} subjects | Match: {set(component['UniqueID']) == eeg_ids}")
print(f"Cluster:   {len(cluster['UniqueID'].unique())} subjects | Match: {set(cluster['UniqueID']) == eeg_ids}")
print(f"Demog:     {len(demog['UniqueID'].unique())} subjects | Match: {set(demog['UniqueID']) == eeg_ids}")
print(f"\nAll four files match: {set(component['UniqueID']) == eeg_ids and set(cluster['UniqueID']) == eeg_ids and set(demog['UniqueID']) == eeg_ids}")


In [ ]:
## Step 5: Merge all files into master dataframe on UniqueID
master = spectral.merge(component, on='UniqueID', how='inner')
master = master.merge(cluster, on='UniqueID', how='inner')
master = master.merge(demog, on='UniqueID', how='inner')

# Verify no subjects lost in merge
print(f"Subjects after merge: {master['UniqueID'].nunique()} (expected 372)")

# Drop N2 columns (unreliable from adolescence onwards)
master = master.drop(columns=['N2_amp_diff', 'N2_lat_diff'])

# Drop duplicate columns from merge
cols_to_drop = ['Group_x', 'Group_y', 
                'group_x', 'group_y', 'Age_Group_x', 'Age_Group_y',
                'schedule_enrol', 'age']
master = master.drop(columns=cols_to_drop)

# Reorder columns - identifiers first
id_cols = ['UniqueID', 'group_original', 'age_group', 'schedule_adj', 'sex', 'age_years']
eeg_cols = [c for c in master.columns if c not in id_cols + ['viq', 'piq', 'fsiq', 'CSS_total', 'SRS_tscore', 'SSP_tactile', 'SSP_audfilt', 'SSP_visualaudisens', 'SSP_total', 'SSP_hype', 'SSP_hypo']]
clin_cols = ['viq', 'piq', 'fsiq', 'CSS_total', 'SRS_tscore', 'SSP_tactile', 'SSP_audfilt', 'SSP_visualaudisens', 'SSP_total', 'SSP_hype', 'SSP_hypo']

master = master[id_cols + eeg_cols + clin_cols]

print(f"Master shape: {master.shape}")
print(master.columns.tolist())

# Save all files
save_path = r'C:\Users\cigde\Dropbox\Habituation_paper\October_Emily\input_arrays\pls_regression'
spectral.to_csv(os.path.join(save_path, 'spectral_RS_final.csv'), index=False)
component.to_csv(os.path.join(save_path, 'component_RS_final.csv'), index=False)
cluster.to_csv(os.path.join(save_path, 'cluster_RS_final.csv'), index=False)
demog.to_csv(os.path.join(save_path, 'demog_clin_data_final.csv'), index=False)
master.to_csv(os.path.join(save_path, 'master_RS.csv'), index=False)
print("All files saved")

In [1]:
## end of scrip ##

# # NOTE: the master file (master_RS.csv) is used for PLS, which needs all 23 EEG features
# (temporal + spectral) for every participant. The 3 participants excluded for abnormal PSD
# have no spectral features, so they are removed from the master file.
# Master = 372 participants. PLS results are done on these 372.